In [4]:
import osmnx as ox
import pandas as pd
import warnings

# Matikan warning agar output bersih (terkait CRS/Geometri)
warnings.filterwarnings("ignore")

# 1. Tentukan Area 
place_name = "Kabupaten Sidoarjo, Jawa Timur, Indonesia"

# 2. Definisikan Tags (UPDATE: Penambahan Fasilitas Umum & Infrastruktur)
tags_fasilitas = {
    # --- PENDIDIKAN ---
    'sekolah': {'amenity': ['school', 'kindergarten', 'university', 'college']},
    
    # --- KESEHATAN ---
    'kesehatan': {'amenity': ['hospital', 'clinic', 'doctors', 'pharmacy', 'dentist']},
    
    # --- TRANSPORTASI UMUM ---
    'transportasi': {
        'amenity': ['bus_station', 'taxi'], 
        'public_transport': ['station', 'stop_position'],
        'railway': ['station', 'halt']
    },
    
    # --- PERBELANJAAN ---
    'belanja': {'shop': ['supermarket', 'convenience', 'department_store', 'mall', 'marketplace']},
    
    # --- IBADAH ---
    'ibadah': {'amenity': ['place_of_worship']},

    # --- FASILITAS UMUM & REKREASI (BARU) ---
    # Mencakup: Alun-alun, Stadion, GOR, Kolam Renang, Tempat Wisata
    'fasilitas_umum': {
        'leisure': ['park', 'stadium', 'sports_centre', 'swimming_pool', 'playground', 'water_park'],
        'sport': ['soccer', 'swimming'],
        'tourism': ['attraction', 'theme_park', 'zoo']
    },

    # --- INFRASTRUKTUR PENTING (BARU) ---
    # Mencakup: Bandara (Juanda), Pintu Tol
    'infrastruktur': {
        'aeroway': ['aerodrome', 'terminal'],       # Bandara
        'barrier': ['toll_booth'],                  # Gerbang Tol
        'highway': ['motorway_junction']            # Exit Tol / Simpang Susun
    }
}

# Fungsi untuk mengambil data
def get_poi_data(place, tags, category_name):
    print(f"Mengambil data kategori: {category_name}...")
    try:
        # Download data
        gdf = ox.features_from_place(place, tags)
        
        # Konversi ke Centroid (Titik Tengah)
        gdf['centroid'] = gdf.geometry.centroid
        gdf['latitude'] = gdf['centroid'].y
        gdf['longitude'] = gdf['centroid'].x
        
        # --- UPDATE PENTING DI SINI ---
        # Kita harus mendaftarkan kolom baru (leisure, aeroway, dll) agar ikut tersimpan
        cols_to_keep = [
            'name', 'amenity', 'shop', 'brand',           # Kolom lama
            'leisure', 'sport', 'tourism',                # Kolom Fasum
            'aeroway', 'highway', 'barrier',              # Kolom Infrastruktur
            'latitude', 'longitude'
        ]
        
        # Filter hanya kolom yang benar-benar ada di hasil download
        available_cols = [c for c in cols_to_keep if c in gdf.columns]
        df_clean = gdf[available_cols].copy()
        
        # Tambah kolom kategori
        df_clean['category_group'] = category_name
        
        return df_clean
    except Exception as e:
        # Error biasanya karena tidak ada data di kategori tersebut
        # print(f"Info: {category_name} kosong atau error kecil.") 
        return pd.DataFrame()

# 3. Loop Eksekusi
all_data = []

for cat, tag in tags_fasilitas.items():
    df = get_poi_data(place_name, tag, cat)
    if not df.empty:
        all_data.append(df)

# Gabung & Simpan
if all_data:
    final_df = pd.concat(all_data, ignore_index=True)
    
    # Hapus data yang namanya kosong (opsional, tapi disarankan agar data bersih)
    final_df = final_df.dropna(subset=['name'])
    
    print(f"\nTotal fasilitas ditemukan: {len(final_df)}")
    
    # Tampilkan sampel data Fasilitas Umum untuk memastikan Alun-alun/Stadion masuk
    print("\nContoh Fasilitas Umum/Infrastruktur yang ditemukan:")
    sample_fasum = final_df[final_df['category_group'].isin(['fasilitas_umum', 'infrastruktur'])]
    print(sample_fasum[['name', 'category_group', 'latitude', 'longitude']].head(10))
    
    # Simpan
    final_df.to_csv("fasilitas_sidoarjo_lengkap_update.csv", index=False)
    print("\nData tersimpan ke 'fasilitas_sidoarjo_lengkap_update.csv'")
else:
    print("Tidak ada data ditemukan.")

Mengambil data kategori: sekolah...
Mengambil data kategori: kesehatan...
Mengambil data kategori: transportasi...
Mengambil data kategori: belanja...
Mengambil data kategori: ibadah...
Mengambil data kategori: fasilitas_umum...
Mengambil data kategori: infrastruktur...

Total fasilitas ditemukan: 648

Contoh Fasilitas Umum/Infrastruktur yang ditemukan:
                                    name  category_group  latitude   longitude
649                             Tulangan  fasilitas_umum -7.465561  112.651019
650                Under The Cherry Tree  fasilitas_umum -7.404263  112.574690
651               Citra Garden Waterpark  fasilitas_umum -7.437339  112.697847
652  Taman Paris Blok B, Puri Surya Jaya  fasilitas_umum -7.390315  112.731894
653                   Taman Pinang Indah  fasilitas_umum -7.455051  112.706425
654                      Taman Vancouver  fasilitas_umum -7.393514  112.735074
655                         Taman Athena  fasilitas_umum -7.393635  112.732528
656         

In [16]:
import osmnx as ox
import pandas as pd
import warnings
import numpy as np

# Matikan warning
warnings.filterwarnings("ignore")

# 1. Tentukan Area 
place_name = "Kabupaten Sidoarjo, Jawa Timur, Indonesia"

# 2. Definisikan Tags 
tags_fasilitas = {
    'sekolah': {'amenity': ['school', 'kindergarten', 'university', 'college', 'childcare']},
    'kesehatan': {'amenity': ['hospital', 'clinic', 'doctors', 'pharmacy', 'dentist']},
    'transportasi': {
        'amenity': ['bus_station', 'taxi'], 
        'public_transport': ['station', 'stop_position'],
        'railway': ['station', 'halt', 'stop']
    },
    'belanja': {'shop': ['supermarket', 'convenience', 'department_store', 'mall', 'marketplace']},
    'ibadah': {'amenity': ['place_of_worship']},
    'fasilitas_umum': {
        'leisure': ['park', 'stadium', 'sports_centre', 'swimming_pool', 'playground', 'water_park', 'garden'],
        'sport': ['soccer', 'swimming'],
        'tourism': ['attraction', 'theme_park', 'zoo']
    },
    'infrastruktur': {
        'aeroway': ['aerodrome', 'terminal'], # Tag untuk Bandara
        'barrier': ['toll_booth'],
        'highway': ['motorway_junction'] 
    }
}

# Fungsi Helper Download
def get_poi_data(place, tags, category_name):
    print(f"Mengambil data kategori: {category_name}...")
    try:
        gdf = ox.features_from_place(place, tags)
        gdf['centroid'] = gdf.geometry.centroid
        gdf['latitude'] = gdf['centroid'].y
        gdf['longitude'] = gdf['centroid'].x
        
        cols_to_keep = [
            'name', 'amenity', 'shop', 'brand', 
            'leisure', 'sport', 'tourism', 
            'aeroway', 'highway', 'barrier', 'railway', 'public_transport', 
            'latitude', 'longitude'
        ]
        available_cols = [c for c in cols_to_keep if c in gdf.columns]
        df_clean = gdf[available_cols].copy()
        df_clean['category_group'] = category_name
        return df_clean
    except Exception:
        return pd.DataFrame()

# 3. Loop Download
all_data = []
for cat, tag in tags_fasilitas.items():
    df = get_poi_data(place_name, tag, cat)
    if not df.empty:
        all_data.append(df)

# ==============================================================================
# 4. LOGIKA KLASIFIKASI SPESIFIK (UPDATE: BANDARA)
# ==============================================================================
def classify_specific(row):
    # Ekstrak data kolom ke string lowercase agar aman
    name = str(row['name']).lower() if pd.notna(row['name']) else ""
    amenity = str(row['amenity']).lower() if 'amenity' in row and pd.notna(row['amenity']) else ""
    shop = str(row['shop']).lower() if 'shop' in row and pd.notna(row['shop']) else ""
    barrier = str(row['barrier']).lower() if 'barrier' in row and pd.notna(row['barrier']) else ""
    highway = str(row['highway']).lower() if 'highway' in row and pd.notna(row['highway']) else ""
    railway = str(row['railway']).lower() if 'railway' in row and pd.notna(row['railway']) else ""
    leisure = str(row['leisure']).lower() if 'leisure' in row and pd.notna(row['leisure']) else ""
    pub_trans = str(row['public_transport']).lower() if 'public_transport' in row and pd.notna(row['public_transport']) else ""
    
    # [PENTING] Ambil kolom aeroway
    aeroway = str(row['aeroway']).lower() if 'aeroway' in row and pd.notna(row['aeroway']) else ""

    # --- INFRASTRUKTUR UTAMA ---

    # 1. [BARU] BANDARA
    # Cek tag aeroway='aerodrome' atau nama mengandung 'bandara'/'juanda'
    if aeroway == 'aerodrome' or 'bandara' in name or 'juanda' in name:
        return 'Bandara'

    # 2. Entry/Exit Tol
    if barrier == 'toll_booth' or highway == 'motorway_junction' or 'gerbang tol' in name or 'pintu tol' in name:
        return 'Entry/Exit Tol'
    
    # --- TRANSPORTASI ---
    if railway == 'station' or 'stasiun' in name:
        return 'Stasiun'
    
    if amenity == 'bus_station' or 'terminal' in name:
        return 'Terminal'
    
    if pub_trans == 'stop_position' or 'halte' in name:
        return 'Halte'
    
    # --- KOMERSIAL ---
    # Mall
    if (shop in ['mall', 'department_store'] or 
        'mall' in name or 'plaza' in name or 'trade center' in name or 
        'suncity' in name or 'transmart' in name or 'lippo' in name or 'ramayana' in name):
        return 'Mall'

    # Supermarket
    if (shop == 'supermarket' or 
        'superindo' in name or 'hypermart' in name or 'grosir' in name or 
        'swalayan' in name or 'hero' in name or 'top market' in name):
        return 'Supermarket'

    # Minimarket
    if (shop == 'convenience' or 
        'indomaret' in name or 'alfamart' in name or 'alfamidi' in name or 
        'circle k' in name or 'mart' in name):
        return 'Minimarket'
    
    # --- SOSIAL ---
    if amenity == 'place_of_worship' or 'masjid' in name or 'gereja' in name or 'pura' in name or 'wihara' in name:
        return 'Tempat Ibadah'
    
    if amenity in ['hospital', 'clinic', 'doctors', 'pharmacy', 'dentist'] or 'puskesmas' in name or 'rumah sakit' in name or 'apotek' in name:
        return 'Fasilitas Kesehatan'
    
    # --- PENDIDIKAN ---
    if (amenity in ['university', 'college'] or 'universitas' in name or 'politeknik' in name or 'poltekkes' in name or 'akademi' in name or 'institut' in name):
        return 'Universitas'

    if ('sma ' in name or ' sma' in name or 'sman' in name or 'smkn' in name or 'smk' in name or 'sekolah menengah atas' in name or 'madrasah aliyah' in name or 'ma ' in name or ' man ' in name):
        return 'SMA/SMK'

    if 'smp' in name or 'sekolah menengah pertama' in name or 'smpn' in name or 'mts ' in name:
        return 'SMP'

    if 'sd ' in name or ' sd' in name or 'sdit' in name or 'sdn' in name or 'sekolah dasar' in name or 'mi ' in name:
        return 'SD'
    
    if (amenity in ['kindergarten', 'childcare'] or 'tk ' in name or 'paud' in name or 'ra ' in name):
        return 'TK'

    # --- REKREASI ---
    if (leisure in ['stadium', 'sports_centre'] or 'stadion' in name or 'gelora' in name or 'gor ' in name):
        return 'Stadion'

    if (leisure in ['park', 'playground', 'garden'] or 'taman' in name or 'alun-alun' in name):
        return 'Taman'

    return 'Lainnya'

# ==============================================================================

# Gabung & Proses
if all_data:
    final_df = pd.concat(all_data, ignore_index=True)
    
    # Isi NaN (Pastikan aeroway masuk list ini)
    cols_check = ['amenity', 'shop', 'barrier', 'highway', 'railway', 'leisure', 'public_transport', 'aeroway']
    for col in cols_check:
        if col not in final_df.columns:
            final_df[col] = np.nan

    print("Sedang mengkategorikan data (termasuk Bandara)...")
    final_df['kategori_spesifik'] = final_df.apply(classify_specific, axis=1)
    
    # Filter: Hapus 'Lainnya' yang Namanya Kosong, tapi pertahankan Halte/Bandara
    final_df = final_df[ (final_df['name'].notna()) | 
                         (final_df['kategori_spesifik'].isin(['Halte', 'Bandara'])) ]

    # Rapikan urutan kolom
    cols = [c for c in final_df.columns if c != 'kategori_spesifik'] + ['kategori_spesifik']
    final_df = final_df[cols]

    print(f"\nTotal fasilitas ditemukan: {len(final_df)}")
    
    # Cek jumlah per kategori
    print("\nJumlah per Kategori Spesifik:")
    print(final_df['kategori_spesifik'].value_counts())
    
    # Cek Sampel Bandara
    print("\nContoh Bandara:")
    sample = final_df[final_df['kategori_spesifik'] == 'Bandara'].head()
    if not sample.empty:
        print(sample[['name', 'aeroway', 'kategori_spesifik']])

    # Simpan
    output_file = "new_fasilitas_sidoarjo_complete_bandara.csv"
    final_df.to_csv(output_file, index=False)
    print(f"\nData tersimpan ke '{output_file}'")
else:
    print("Tidak ada data ditemukan.")

Mengambil data kategori: sekolah...
Mengambil data kategori: kesehatan...
Mengambil data kategori: transportasi...
Mengambil data kategori: belanja...
Mengambil data kategori: ibadah...
Mengambil data kategori: fasilitas_umum...
Mengambil data kategori: infrastruktur...
Sedang mengkategorikan data (termasuk Bandara)...

Total fasilitas ditemukan: 648

Jumlah per Kategori Spesifik:
kategori_spesifik
Tempat Ibadah          162
Minimarket              86
SD                      52
Taman                   48
SMA/SMK                 42
Fasilitas Kesehatan     41
Entry/Exit Tol          36
Lainnya                 34
Halte                   31
SMP                     30
Supermarket             19
TK                      14
Stasiun                 13
Universitas             12
Mall                     8
Terminal                 7
Stadion                  7
Bandara                  6
Name: count, dtype: int64

Contoh Bandara:
                          name   aeroway kategori_spesifik
727       